# FreshCheck Colab Runner

ใช้ notebook นี้เป็นตัวกลางสำหรับ `GitHub + Colab + Google Drive`

Workflow:
1. Clone หรือ update repo จาก GitHub
2. Mount Google Drive
3. ติดตั้ง dependencies
4. รัน CLI `run_freshcheck.py`


In [ ]:
#@title 1) Clone / Update repository from GitHub
REPO_URL = "https://github.com/techasit239/DADS7202_PigPicture.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
REPO_DIR = "/content/DADS7202_PigPicture"  #@param {type:"string"}

import os
import subprocess
from pathlib import Path

repo_path = Path(REPO_DIR)
if repo_path.exists():
    print(f"[INFO] Repo exists at {repo_path}. Pulling latest changes from {BRANCH}...")
    subprocess.run(["git", "-C", str(repo_path), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(repo_path), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(repo_path), "pull", "origin", BRANCH], check=True)
else:
    print(f"[INFO] Cloning {REPO_URL} -> {repo_path}")
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(repo_path)], check=True)

%cd /content/DADS7202_PigPicture


In [ ]:
#@title 2) Mount Google Drive and define project paths
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/FreshCheck')  #@param {type:"string"}
KAGGLE_DATA_DIR = DRIVE_ROOT / 'data' / 'kaggle_meat'
THAI_DATA_DIR = DRIVE_ROOT / 'data' / 'thai_retail'
CVAT_XML_PATH = DRIVE_ROOT / 'data' / 'thai_retail_cvat_annotations.xml'
ARTIFACTS_DIR = DRIVE_ROOT / 'artifacts'

for folder in [DRIVE_ROOT, ARTIFACTS_DIR, ARTIFACTS_DIR / 'splits', ARTIFACTS_DIR / 'train', ARTIFACTS_DIR / 'eval', ARTIFACTS_DIR / 'predict', ARTIFACTS_DIR / 'phase2']:
    folder.mkdir(parents=True, exist_ok=True)

print('[OK] Drive mounted and directories prepared')
print('KAGGLE_DATA_DIR =', KAGGLE_DATA_DIR)
print('THAI_DATA_DIR   =', THAI_DATA_DIR)
print('CVAT_XML_PATH   =', CVAT_XML_PATH)
print('ARTIFACTS_DIR   =', ARTIFACTS_DIR)


In [ ]:
#@title 3) Install dependencies
%cd /content/DADS7202_PigPicture
!python -m pip install -q --upgrade pip
!python -m pip install -q -r requirements.txt


In [ ]:
#@title 4) Prepare group-aware Kaggle train/val splits
%cd /content/DADS7202_PigPicture
!python run_freshcheck.py prepare-splits --data-dir "$KAGGLE_DATA_DIR" --output-dir "$ARTIFACTS_DIR/splits"


In [ ]:
#@title 5) Train classifiers from the same CLI
MODELS = "all"  #@param ["all", "efficientnet_b0", "swin_t"]
EPOCHS = 15  #@param {type:"integer"}
BATCH_SIZE = 16  #@param {type:"integer"}
NUM_WORKERS = 2  #@param {type:"integer"}
PRETRAINED = True  #@param {type:"boolean"}

%cd /content/DADS7202_PigPicture
!python run_freshcheck.py train --train-csv "$ARTIFACTS_DIR/splits/kaggle_train.csv" --val-csv "$ARTIFACTS_DIR/splits/kaggle_val.csv" --output-dir "$ARTIFACTS_DIR/train" --models $MODELS --epochs $EPOCHS --batch-size $BATCH_SIZE --num-workers $NUM_WORKERS {'--pretrained' if PRETRAINED else '--no-pretrained'}


In [ ]:
#@title 6) Evaluate checkpoints on validation CSV
MODELS = "all"  #@param ["all", "efficientnet_b0", "swin_t"]
BATCH_SIZE = 16  #@param {type:"integer"}
NUM_WORKERS = 2  #@param {type:"integer"}

%cd /content/DADS7202_PigPicture
!python run_freshcheck.py evaluate --csv "$ARTIFACTS_DIR/splits/kaggle_val.csv" --checkpoint-dir "$ARTIFACTS_DIR/train/checkpoints" --output-dir "$ARTIFACTS_DIR/eval" --models $MODELS --batch-size $BATCH_SIZE --num-workers $NUM_WORKERS


In [ ]:
#@title 7) Predict on one image or one folder
INPUT_PATH = "/content/drive/MyDrive/FreshCheck/inference_samples"  #@param {type:"string"}
MODELS = "all"  #@param ["all", "efficientnet_b0", "swin_t"]
BATCH_SIZE = 16  #@param {type:"integer"}

%cd /content/DADS7202_PigPicture
!python run_freshcheck.py predict --input-path "$INPUT_PATH" --checkpoint-dir "$ARTIFACTS_DIR/train/checkpoints" --output-dir "$ARTIFACTS_DIR/predict" --models $MODELS --batch-size $BATCH_SIZE


In [ ]:
#@title 8) Phase 2 foundation: CVAT XML -> GT masks + manifest CSV
%cd /content/DADS7202_PigPicture
!python run_freshcheck.py prepare-cvat --thai-data-dir "$THAI_DATA_DIR" --cvat-xml-path "$CVAT_XML_PATH" --output-dir "$ARTIFACTS_DIR/phase2"


## Notes

- ถ้าแก้โค้ดใน GitHub แล้ว ให้กลับไปรัน cell `1) Clone / Update repository from GitHub`
- ไฟล์ model checkpoints จะถูกเก็บใน `MyDrive/FreshCheck/artifacts/train/checkpoints`
- ไฟล์ใหญ่ทั้งหมดควรอยู่บน Drive ไม่ควร commit ลง GitHub
- โมเดล gated (`SAM3`, `Florence-2`, `DINOv3`) ยังต้องเปิด access/token ก่อนค่อยทำ runner เพิ่ม
